# GPT
We have everything we need now to simply assemble a language model, but we also need to add two things that the architecture alone doesn't cover: how to compute loss during training, and how to generate text at inference.

The full pipeline:
raw tokens → embedding → N transformer blocks → RMSNorm → logits

During training: logits → cross-entropy loss → backprop

During inference: logits → sample next token → append → repeat

## Pipeline
We just need create a GPT class to implement the full pipeline. We start with a configuration class, which includes our parameters and hyperparameters.

Then, we implement the GPT class itself and simply execute the pipeline in the order that we described above. Note that in addition to normalizing by means of RMSNorm before each attention and each FFN block, we also do one last normalization after our tranformer blocks. Because it is the last block, we do not normalize again (as we do at the start of attention). In other words, the final RMSNorm substitutes for the normalization that would have happened at the start of the next transformer block, had there been one. So it prevents a potential drift in scale that can occur as it passes through the last transformer sublayer. This is be helpful as we complete one final projection via an unembedding matrix.

### Weight Initialization
A notion that we have been referencing throughout is that of random initialization: when we create the model, all weights start random. However, we still need to be thoughtful about exactly how we randomize our weights. In particular, we need to be thoughtful about the scale of randomness, because when weights are too small, gradients can disappear. And when they are too large, activations explode (even despite our best efforts of scaling and normalization). Empirically (in the GPT-2 paper), we found that initializing via N(0,0.02) provides a good balance.

### Logits
The unembedding matrix ($W_U$) will be of shape embeddings_dim * vocab_size. We multiply by the output of our transformer layers to get a matrix of batch * seq_len * vocab_size. These logits are the input to softmax, which we know will sum to one. This ultimately answers the question: for a given vocab word, how likely is it to be the next word in the sequence? Note that logits are per position: each position in the sequence gets its own distribution over vocab (note the last two dimensions!)

The logits then feed into loss during training and sampling during inference.

### Loss
We include loss inside the model. This is a design choice and is often done for simplicity. After ```lm_head```, our logits have shape batch * seq_len * vocab_size. The function we use to compute loss, ```F.cross_entropy``` expects (N,C) where C is the number of classes. So before passing them into cross entropy, we need to flatten our logits to (batch * seq_len) * vocab_size.

$L = -log(P(correct token))$

$L_{total} = \frac{-1}{N}∑log(P_{correct})$ where N = batch * (seq_len - 1).

so if the correct token is id=4 and our softmax gives probability 0.05, loss = -log(.05) = 3.0. Note that we are not directly penalized for wrong predictions.

We use log because the gradient is smoother and magnifies errors. Think about a log function: smaller values are magnified. So when a model is confident and wrong, the gradient passed back is larger, thus allowing the model to learn faster.

## Inference
At a high level, inference is just a loop:
1. Prompt
2. Forward pass -> apply temp -> softmax -> sample token
3. Append token until EOS or max_new_tokens

Most of these we have already covered. But two concepts that we have not:
1. Temperature: At inference time, we modify our logits by dividing by temperature so that our softmax formula is:

$σ(z)_i = \frac{e^{\frac{z_i}{T}}}{∑e^{\frac{z_j}{T}}}$

With higher temperatures, our distribution is more spread out / random. Think:

$ lim_{T→∞} \frac{e^{\frac{z_i}{T}}}{∑e^{\frac{z_j}{T}}} = \frac{1}{K}$ where K=vocab_size.

With lower temperatures, our distribution is more deterministic:

$\lim_{T \to 0} \frac{e^{z_i/T}}{\sum e^{z_j/T}} = \begin{cases} 1 & \text{if } i = \arg\max_j z_j \\ 0 & \text{otherwise} \end{cases}$

When temp = 1, softmax is unchanged.

2. Top-k selection: Selecting one token out of vocab_size (50257) can lead to some bad selections, so we limit the model to choosing from the k most likely tokens. We just have to make sure we re-normalize to ensure the probabilities sum to 1.

## Quick note on weight tying:
In building nanoGPT, Karpathy used the same set of weights in the Embeddings lookup table and the Unembedding table. This is the standard optimized procedure in LLMs- we could use two matrices, but since they functionally convey the same information, it is optimal to reuse this matrix. As Raiyan points out, there are several tangible benefits:
1. we save space (50257*768 = 38.6M params)
2. better regularization: shared matrix gets gradient signals from both directions

```
self.lm_head = nn.Linear(config.embeddings_dim, config.vocab_size, bias = False)
self.embeddings.embed.weight = self.lm_head.weight
```
A quick note on the code:```self.lm_head``` performs a matrix multiplication operation (x @ $W_U$) but setting ```self.embeddings.embed.weight = self.lm_head.weight``` actually makes sure that the weights ```self.lm_head``` points to the same set of learnable weights used by the embeddings matrix.

## Mixed Precision
Normally, weights and activations are stored as float32. During the forward pass, we do not need as much precision, so we use float16, which uses half the amount of memory. They also run faster on GPUs. But weight updates DO need precision or else we would lose too much memory. So mixed precision does:
* Forward pass: 16 bits per number
* UpdatesL 32 bits per number

```
# 1. Before the loop — handles gradient scaling
scaler = torch.amp.GradScaler('cuda')

# 2. Forward pass in bfloat16
with torch.amp.autocast('cuda'):
    logits, loss = model.forward(input_ids, target_ids)

# 3. Scale loss up before backward (prevents underflow)
scaler.scale(loss).backward()

# 4. Unscale before gradient clipping
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.)

# 5. Unscale gradients back, then update weights
scaler.step(optimizer)
scaler.update()
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [8]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
  # Params
  vocab_size : int = 50257
  embeddings_dim : int = 768
  head_count : int = 12
  transformer_count : int = 12
  max_seq_len : int = 2048

  # Regularization
  embd_dropout : int = 0.1
  dropout : int = 0.1
  layer_norm_epsilon : int = 1e-5

  # Training
  weight_decay : int = 0.1
  learning_rate : int = 1e-4
  warmup_steps : int  = 2000
  max_steps : int = 100000
  batch_size : int = 64
  grad_accum_steps : int = 4
  beta1 : int = 0.9
  beta2 : int = 0.95
  eps : int = 1e-8


In [9]:
class GPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.config = config

    self.embeddings = Embedding(config.vocab_size, config.embeddings_dim)

    self.embd_dropout = nn.Dropout(config.embd_dropout)

    self.layers = nn.ModuleList([TransformerBlock(config.embeddings_dim,
                                                  config.head_count,
                                                  config.dropout) for _ in range(config.transformer_count)])

    self.RMSNorm = RMSNorm(config.embeddings_dim)

    self.lm_head = nn.Linear(config.embeddings_dim, config.vocab_size, bias = False)

    # weight tying
    self.embeddings.embed.weight = self.lm_head.weight

    self.apply(self._init_weights)
    print(f"GPT initialized with {self._count_params():,} parameters")

  # helper
  def _init_weights(self, module: nn.Module):
      if isinstance(module, nn.Linear) or isinstance(module, nn.Embedding):
          torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
          if module.bias is not None:
              torch.nn.init.zeros_(module.bias)
  # for fun
  def _count_params(self):
    return sum(p.numel() for p in self.parameters())

  def make_mask(self, seq_len) -> torch.Tensor:
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

  def forward(self, input_ids: torch.Tensor, targets: torch.Tensor = None) -> tuple:

    batch_size, seq_len = input_ids.shape

    x = self.embeddings(input_ids)
    x = self.embd_dropout(x)

    mask = self.make_mask(seq_len)

    for layer in self.layers:
      x = layer(x, mask)

    x = self.RMSNorm(x)
    logits = self.lm_head(x)

    # ====== LOSS =======
    loss = None

    if targets is not None:
      # we need to shift logits and targets so that position x of logits predicts x + 1 of targets
      # But already shifted in TextDataset
      # logits = logits[:, :-1, :].contiguous()
      # targets = targets[:, 1:].contiguous()


      reshaped_logits = logits.view(-1, logits.shape[-1])
      reshaped_targets = targets.view(-1)
      loss = F.cross_entropy(reshaped_logits, reshaped_targets)

    # shape: [batch * seq_len - 1 * vocab_size]
    return (logits, loss)

  @torch.no_grad()
  def generate(self, input_ids: torch.Tensor, max_new_tokens: int, temperature: float = 1.0, top_k : int = None, top_p: float = None):

    # switches model to evaluation mode
    self.eval()

    for _ in range(max_new_tokens):
      if input_ids.shape[1] > self.config.max_seq_len:
        input_ids = input_ids[:, -self.config.max_seq_len:]

      logits, _ = self(input_ids)
      """
      After the forward pass, logits are [batch, seq_len, vocab_size]. We only care about the last position: the next predicted token.
      So the [:,-1,:] selects the last position.
      """
      last = logits[:, -1, :] #batch * vocab_size

      last = last / temperature

      if top_k is not None:
        v, _ = torch.topk(last, top_k)
        last[last < v[:, [-1]]] = float('-inf') #batch * top_k

      # skipping nucleus filtering
      # sampling next token
      probs = F.softmax(last, dim = -1)
      next_token = torch.multinomial(probs, num_samples = 1)

      # append to sequence
      input_ids = torch.cat((input_ids, next_token), dim = 1)

    return input_ids
